# 02. Transformer from Scratch ★

## 학습 목표
- "Attention Is All You Need" 논문의 아키텍처를 **완전히** 구현
- Multi-Head Attention, FFN, LayerNorm, Residual Connection 이해
- Encoder-Decoder 전체 구조를 직접 조립
- 간단한 task로 학습 테스트

## 핵심 논문
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) (Vaswani et al., 2017)

## 구현 순서
1. Multi-Head Attention
2. Position-wise FFN
3. Layer Normalization
4. Residual Connection
5. Encoder Block → Encoder
6. Decoder Block → Decoder
7. 전체 Transformer 조립
8. Copy Task로 테스트

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Multi-Head Attention

### 왜 여러 Head가 필요한가?

Single-head attention은 하나의 "관점"으로만 관계를 파악한다.
하지만 언어에는 여러 종류의 관계가 동시에 존재한다:

| Head | 학습할 수 있는 관계 |
|------|-------------------|
| Head 1 | 구문적 관계 (주어-동사) |
| Head 2 | 의미적 관계 (동의어, 반의어) |
| Head 3 | 위치적 관계 (인접 단어) |
| Head 4 | 장거리 의존성 |

### 수식

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

- 전체 $d_{model}$을 $h$개 head로 나눈다: $d_k = d_{model} / h$
- 각 head가 독립적으로 attention 수행
- 결과를 concat 후 $W^O$로 합침

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Attention (Vaswani et al., 2017)"""
    
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # 각 head의 차원
        
        # Q, K, V 프로젝션 (한번에 모든 head용)
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        
        # 출력 프로젝션
        self.W_O = nn.Linear(d_model, d_model)
    
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        Args:
            Q: (batch, n_heads, seq_len, d_k)
            K: (batch, n_heads, seq_len, d_k)
            V: (batch, n_heads, seq_len, d_k)
            mask: (batch, 1, 1, seq_len) or (batch, 1, seq_len, seq_len)
        """
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        weights = F.softmax(scores, dim=-1)
        output = torch.matmul(weights, V)
        return output, weights
    
    def forward(self, Q, K, V, mask=None):
        """
        Args:
            Q: (batch, seq_len_q, d_model)
            K: (batch, seq_len_k, d_model)
            V: (batch, seq_len_k, d_model)
        Returns:
            output: (batch, seq_len_q, d_model)
            weights: (batch, n_heads, seq_len_q, seq_len_k)
        """
        batch_size = Q.size(0)
        
        # 1. Linear projection
        Q = self.W_Q(Q)  # (batch, seq_len, d_model)
        K = self.W_K(K)
        V = self.W_V(V)
        
        # 2. Split into n_heads
        # (batch, seq_len, d_model) → (batch, seq_len, n_heads, d_k) → (batch, n_heads, seq_len, d_k)
        Q = Q.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # 3. Attention
        output, weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # 4. Concat heads
        # (batch, n_heads, seq_len, d_k) → (batch, seq_len, n_heads, d_k) → (batch, seq_len, d_model)
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # 5. Output projection
        output = self.W_O(output)
        
        return output, weights


# 테스트
d_model = 64
n_heads = 8
seq_len = 10
batch_size = 2

mha = MultiHeadAttention(d_model, n_heads)
x = torch.randn(batch_size, seq_len, d_model)

output, weights = mha(x, x, x)  # Self-Attention: Q=K=V=x
print(f"입력 shape:  {x.shape}")
print(f"출력 shape:  {output.shape}")
print(f"가중치 shape: {weights.shape} (batch, n_heads, seq_q, seq_k)")
print(f"\nd_model={d_model}, n_heads={n_heads}, d_k={d_model//n_heads}")
print(f"→ 각 head는 {d_model//n_heads}차원으로 독립적으로 attention 수행")

In [ ]:
# 각 head의 attention 패턴 시각화

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

w = weights[0].detach().numpy()  # 첫 번째 배치

for i in range(n_heads):
    im = axes[i].imshow(w[i], cmap='Blues', aspect='auto')
    axes[i].set_title(f'Head {i+1}', fontsize=11)
    axes[i].set_xlabel('Key')
    axes[i].set_ylabel('Query')

plt.suptitle('Multi-Head Attention: 각 Head의 Attention Pattern', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("→ 각 head가 서로 다른 패턴을 학습한다")
print("  (초기화 직후이므로 아직 의미 있는 패턴은 아님)")

---
## 2. Position-wise Feed-Forward Network (FFN)

Attention 이후 각 위치별로 독립적으로 적용하는 2층 네트워크.

$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

- 내부 차원($d_{ff}$)은 보통 $d_{model}$의 4배
- 왜 필요한가? Attention은 **선형** 변환만 수행 → FFN이 **비선형** 변환 추가
- "Position-wise": 각 토큰 위치에 **동일한 가중치**로 독립 적용

In [ ]:
class PositionwiseFFN(nn.Module):
    """Position-wise Feed-Forward Network"""
    
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = self.linear1(x)       # (batch, seq_len, d_ff)
        x = F.relu(x)             # 비선형 활성화
        x = self.dropout(x)
        x = self.linear2(x)       # (batch, seq_len, d_model)
        return x


# 테스트
d_model = 64
d_ff = 256  # 보통 4 * d_model

ffn = PositionwiseFFN(d_model, d_ff)
x = torch.randn(2, 10, d_model)
output = ffn(x)

print(f"입력 shape:  {x.shape}")
print(f"출력 shape:  {output.shape}")
print(f"\n파라미터 수: {sum(p.numel() for p in ffn.parameters()):,}")
print(f"  linear1: {d_model}x{d_ff} + {d_ff} = {d_model*d_ff + d_ff:,}")
print(f"  linear2: {d_ff}x{d_model} + {d_model} = {d_ff*d_model + d_model:,}")

---
## 3. Layer Normalization

### BatchNorm vs LayerNorm

| | BatchNorm | LayerNorm |
|---|-----------|----------|
| 정규화 방향 | 배치 방향 (같은 feature, 여러 샘플) | 특성 방향 (같은 샘플, 여러 feature) |
| 배치 크기 의존 | O (작은 배치에서 불안정) | X (배치 크기 무관) |
| 시퀀스 길이 | 가변 길이에서 문제 | 가변 길이 OK |
| 주 사용처 | CNN (이미지) | Transformer (NLP) |

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

- $\mu, \sigma^2$: 각 샘플의 마지막 차원(feature)에 대한 평균, 분산
- $\gamma, \beta$: 학습 가능한 스케일/시프트 파라미터

In [ ]:
# LayerNorm 직접 구현 vs PyTorch

class LayerNorm(nn.Module):
    """Layer Normalization 직접 구현"""
    
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))   # 스케일
        self.beta = nn.Parameter(torch.zeros(d_model))   # 시프트
        self.eps = eps
    
    def forward(self, x):
        # x: (batch, seq_len, d_model)
        mean = x.mean(dim=-1, keepdim=True)     # 마지막 차원(feature)에 대해
        std = x.std(dim=-1, keepdim=True)
        normalized = (x - mean) / (std + self.eps)
        return self.gamma * normalized + self.beta


# 비교: 직접 구현 vs PyTorch
d_model = 8
x = torch.randn(1, 3, d_model)

my_ln = LayerNorm(d_model)
pt_ln = nn.LayerNorm(d_model)

my_out = my_ln(x)
pt_out = pt_ln(x)

print(f"입력 x[0,0]: {x[0,0].detach()}")
print(f"  평균: {x[0,0].mean():.4f}, 분산: {x[0,0].var():.4f}")
print(f"\n직접 구현 결과: {my_out[0,0].detach()}")
print(f"  평균: {my_out[0,0].mean():.6f}, 분산: {my_out[0,0].var():.4f}")
print(f"\nPyTorch 결과:  {pt_out[0,0].detach()}")
print(f"  평균: {pt_out[0,0].mean():.6f}, 분산: {pt_out[0,0].var():.4f}")
print(f"\n→ 정규화 후 평균 ≈ 0, 분산 ≈ 1")

In [ ]:
# BatchNorm vs LayerNorm 시각화

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 시각적 설명을 위한 3D 텐서 (batch=4, seq=3, features=5)
data = torch.randn(4, 3, 5)

# 텐서 구조
labels = ['Batch\n(samples)', 'Sequence\n(tokens)', 'Features\n(d_model)']
axes[0].text(0.5, 0.5, f'Tensor: ({4}, {3}, {5})\n\nbatch x seq x features',
             ha='center', va='center', fontsize=14,
             transform=axes[0].transAxes)
axes[0].set_title('Input Tensor Shape', fontsize=13)
axes[0].axis('off')

# BatchNorm: batch 방향으로 정규화
bn_data = np.random.randn(4, 5)
axes[1].imshow(np.ones((4, 5)), cmap='Reds', alpha=0.3, aspect='auto')
axes[1].set_title('BatchNorm\n(batch 방향 정규화)', fontsize=12)
axes[1].set_xlabel('Features')
axes[1].set_ylabel('Batch samples')
for j in range(5):
    axes[1].axvline(x=j-0.5, color='red', linewidth=2, alpha=0.5)
axes[1].text(2, -0.8, '각 feature별로\nbatch 전체의 mean/std 계산',
             ha='center', fontsize=10)

# LayerNorm: feature 방향으로 정규화
axes[2].imshow(np.ones((4, 5)), cmap='Blues', alpha=0.3, aspect='auto')
axes[2].set_title('LayerNorm\n(feature 방향 정규화)', fontsize=12)
axes[2].set_xlabel('Features')
axes[2].set_ylabel('Batch samples')
for i in range(4):
    axes[2].axhline(y=i-0.5, color='blue', linewidth=2, alpha=0.5)
axes[2].text(2, -0.8, '각 샘플별로\nfeature 전체의 mean/std 계산',
             ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("BatchNorm: 세로(↓) 방향으로 정규화 → 배치 크기에 의존")
print("LayerNorm: 가로(→) 방향으로 정규화 → 각 샘플 독립, 배치 무관")

---
## 4. Residual Connection

$$\text{output} = \text{LayerNorm}(x + \text{Sublayer}(x))$$

### 왜 필요한가?

1. **Gradient 흐름**: 깊은 네트워크에서 gradient가 잘 전파되도록
2. **학습 안정성**: 레이어를 거칠 때 "최소한 입력은 보존"됨
3. **점진적 학습**: sublayer가 "변화량(residual)"만 학습하면 됨

In [ ]:
class ResidualConnection(nn.Module):
    """Residual Connection + Layer Normalization"""
    
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, sublayer):
        """
        x + Sublayer(LayerNorm(x))
        (Pre-LN 방식: 원래 논문은 Post-LN이지만, Pre-LN이 더 안정적)
        """
        return x + self.dropout(sublayer(self.norm(x)))


# Residual 효과 확인
d_model = 64
residual = ResidualConnection(d_model)
x = torch.randn(1, 5, d_model)

# sublayer가 0을 반환해도 입력이 보존됨
zero_sublayer = lambda x: torch.zeros_like(x)
output = residual(x, zero_sublayer)
print(f"Sublayer가 0을 반환해도:")
print(f"  입력과 출력이 같은가? {torch.allclose(x, output, atol=1e-5)}")
print(f"  → Residual connection 덕분에 입력이 보존됨!")

---
## 5. Encoder Block 구현

Encoder Block = Multi-Head Self-Attention + FFN (각각 Residual + LayerNorm)

```
Input
  │
  ├──→ LayerNorm → Multi-Head Attention ─→ (+) ── Residual
  │                                        │
  ├──→ LayerNorm → Feed-Forward ──────────→ (+) ── Residual
  │                                        │
  └──────────────────────────────────────→ Output
```

In [ ]:
class EncoderBlock(nn.Module):
    """Transformer Encoder Block"""
    
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PositionwiseFFN(d_model, d_ff, dropout)
        self.residual1 = ResidualConnection(d_model, dropout)
        self.residual2 = ResidualConnection(d_model, dropout)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: (batch, seq_len, d_model)
            mask: padding mask
        Returns:
            output: (batch, seq_len, d_model)
        """
        # Self-Attention + Residual
        x = self.residual1(x, lambda x: self.self_attn(x, x, x, mask)[0])
        
        # FFN + Residual
        x = self.residual2(x, self.ffn)
        
        return x


class Encoder(nn.Module):
    """Transformer Encoder: N개의 EncoderBlock 스택"""
    
    def __init__(self, d_model, n_heads, d_ff, n_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)  # 최종 LayerNorm
    
    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)


# 테스트
d_model = 64
n_heads = 8
d_ff = 256
n_layers = 6

encoder = Encoder(d_model, n_heads, d_ff, n_layers)
x = torch.randn(2, 10, d_model)
output = encoder(x)

print(f"Encoder 구조:")
print(f"  d_model={d_model}, n_heads={n_heads}, d_ff={d_ff}, n_layers={n_layers}")
print(f"  입력: {x.shape} → 출력: {output.shape}")
print(f"  파라미터 수: {sum(p.numel() for p in encoder.parameters()):,}")

---
## 6. Decoder Block 구현

Decoder Block은 Encoder Block보다 하나의 sublayer가 더 있다:

1. **Masked Self-Attention**: 미래 토큰을 볼 수 없도록 causal mask 적용
2. **Cross-Attention**: Encoder 출력(K, V)에 대해 Decoder의 Q로 attention
3. **FFN**: Encoder와 동일

```
Target Input
  │
  ├──→ LayerNorm → Masked Self-Attention ───→ (+)
  │                                           │
  ├──→ LayerNorm → Cross-Attention ─────────→ (+)  ← Encoder Output (K, V)
  │                                           │
  ├──→ LayerNorm → Feed-Forward ────────────→ (+)
  │                                           │
  └─────────────────────────────────────────→ Output
```

In [ ]:
class DecoderBlock(nn.Module):
    """Transformer Decoder Block"""
    
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.masked_self_attn = MultiHeadAttention(d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PositionwiseFFN(d_model, d_ff, dropout)
        self.residual1 = ResidualConnection(d_model, dropout)
        self.residual2 = ResidualConnection(d_model, dropout)
        self.residual3 = ResidualConnection(d_model, dropout)
    
    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        """
        Args:
            x: (batch, tgt_seq_len, d_model) - Decoder 입력
            enc_output: (batch, src_seq_len, d_model) - Encoder 출력
            src_mask: 소스 padding mask
            tgt_mask: 타겟 causal + padding mask
        """
        # 1. Masked Self-Attention (미래를 볼 수 없음)
        x = self.residual1(x, lambda x: self.masked_self_attn(x, x, x, tgt_mask)[0])
        
        # 2. Cross-Attention (Q=Decoder, K=V=Encoder)
        x = self.residual2(x, lambda x: self.cross_attn(x, enc_output, enc_output, src_mask)[0])
        
        # 3. FFN
        x = self.residual3(x, self.ffn)
        
        return x


class Decoder(nn.Module):
    """Transformer Decoder: N개의 DecoderBlock 스택"""
    
    def __init__(self, d_model, n_heads, d_ff, n_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        return self.norm(x)


# 테스트
decoder = Decoder(d_model, n_heads, d_ff, n_layers)
tgt = torch.randn(2, 8, d_model)
enc_out = torch.randn(2, 10, d_model)

dec_output = decoder(tgt, enc_out)
print(f"Decoder 입력: tgt={tgt.shape}, enc_output={enc_out.shape}")
print(f"Decoder 출력: {dec_output.shape}")
print(f"파라미터 수: {sum(p.numel() for p in decoder.parameters()):,}")

---
## 7. 전체 Transformer 조립

지금까지 만든 모든 컴포넌트를 합쳐서 완전한 Transformer를 만든다.

추가로 필요한 것:
- **Token Embedding**: 토큰 ID → 벡터
- **Positional Encoding**: 위치 정보 추가 (다음 노트북에서 자세히)
- **Output Linear + Softmax**: 최종 토큰 예측

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal Positional Encoding (간단 버전, 자세한 것은 03에서)"""
    
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)  # 짝수 인덱스
        pe[:, 1::2] = torch.cos(position * div_term)  # 홀수 인덱스
        
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [ ]:
class Transformer(nn.Module):
    """
    Full Transformer (Vaswani et al., 2017)
    
    완전한 Encoder-Decoder Transformer
    """
    
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=64,
                 n_heads=8, d_ff=256, n_layers=2, dropout=0.1, max_len=100):
        super().__init__()
        
        # Embedding
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        # Encoder & Decoder
        self.encoder = Encoder(d_model, n_heads, d_ff, n_layers, dropout)
        self.decoder = Decoder(d_model, n_heads, d_ff, n_layers, dropout)
        
        # Output projection
        self.output_linear = nn.Linear(d_model, tgt_vocab_size)
        
        self.d_model = d_model
        
        # 가중치 초기화
        self._init_weights()
    
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def encode(self, src, src_mask=None):
        src = self.src_embedding(src) * math.sqrt(self.d_model)
        src = self.positional_encoding(src)
        return self.encoder(src, src_mask)
    
    def decode(self, tgt, enc_output, src_mask=None, tgt_mask=None):
        tgt = self.tgt_embedding(tgt) * math.sqrt(self.d_model)
        tgt = self.positional_encoding(tgt)
        return self.decoder(tgt, enc_output, src_mask, tgt_mask)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        """
        Args:
            src: (batch, src_len) - 소스 토큰 ID
            tgt: (batch, tgt_len) - 타겟 토큰 ID
        Returns:
            logits: (batch, tgt_len, tgt_vocab_size)
        """
        enc_output = self.encode(src, src_mask)
        dec_output = self.decode(tgt, enc_output, src_mask, tgt_mask)
        logits = self.output_linear(dec_output)
        return logits


# Transformer 생성
vocab_size = 20
model = Transformer(
    src_vocab_size=vocab_size,
    tgt_vocab_size=vocab_size,
    d_model=64,
    n_heads=8,
    d_ff=256,
    n_layers=2,
    dropout=0.1
)

print(f"Transformer 전체 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
print(f"\n모델 구조:")
for name, param in model.named_parameters():
    if 'layers.0' in name or 'embedding' in name or 'output' in name:
        if 'layers.0' not in name or 'self_attn.W_Q' in name or 'ffn.linear1' in name:
            print(f"  {name}: {param.shape}")

In [ ]:
# 마스크 생성 유틸리티

def create_masks(src, tgt, pad_idx=0):
    """
    소스와 타겟에 대한 마스크 생성
    
    Args:
        src: (batch, src_len)
        tgt: (batch, tgt_len)
        pad_idx: 패딩 토큰 인덱스
    Returns:
        src_mask: (batch, 1, 1, src_len)
        tgt_mask: (batch, 1, tgt_len, tgt_len)
    """
    # 소스 패딩 마스크
    src_mask = (src != pad_idx).unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, src_len)
    
    # 타겟: 패딩 마스크 + causal 마스크
    tgt_pad_mask = (tgt != pad_idx).unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, tgt_len)
    tgt_len = tgt.size(1)
    tgt_causal_mask = torch.tril(torch.ones(tgt_len, tgt_len, device=tgt.device)).bool()  # (tgt_len, tgt_len)
    tgt_mask = tgt_pad_mask & tgt_causal_mask.unsqueeze(0).unsqueeze(0)  # (batch, 1, tgt_len, tgt_len)
    
    return src_mask, tgt_mask


# Forward pass 테스트
src = torch.randint(1, vocab_size, (2, 8))  # 소스 시퀀스
tgt = torch.randint(1, vocab_size, (2, 6))  # 타겟 시퀀스

src_mask, tgt_mask = create_masks(src, tgt)
logits = model(src, tgt, src_mask, tgt_mask)

print(f"src shape: {src.shape}")
print(f"tgt shape: {tgt.shape}")
print(f"logits shape: {logits.shape} (batch, tgt_len, vocab_size)")
print(f"\n→ 각 타겟 위치에서 vocab_size={vocab_size}개 토큰에 대한 확률을 예측")

---
## 8. Copy Task로 테스트

Transformer가 정말 동작하는지 확인하기 위한 가장 간단한 task:

**Copy Task**: 입력 시퀀스를 그대로 출력으로 복사

```
입력: [3, 7, 2, 5, 1]
출력: [3, 7, 2, 5, 1]
```

이것도 못하면 모델 구현에 문제가 있는 것!

In [ ]:
# Copy Task 데이터 생성

def generate_copy_data(batch_size, seq_len, vocab_size, pad_idx=0, sos_idx=1):
    """
    Copy task 데이터 생성
    src: [3, 7, 2, 5]    (랜덤 토큰)
    tgt: [1, 3, 7, 2, 5] (SOS + src)
    """
    # 2~vocab_size-1 사이의 랜덤 토큰 (0=PAD, 1=SOS 제외)
    src = torch.randint(2, vocab_size, (batch_size, seq_len))
    
    # 타겟: SOS 토큰 + 소스 복사
    sos = torch.full((batch_size, 1), sos_idx)
    tgt = torch.cat([sos, src], dim=1)  # (batch, seq_len + 1)
    
    return src, tgt


# 데이터 예시
src, tgt = generate_copy_data(batch_size=3, seq_len=5, vocab_size=10)
print(f"소스 (입력):  {src}")
print(f"타겟 (출력):  {tgt}")
print(f"\n타겟의 첫 토큰은 항상 SOS(=1)")
print(f"나머지는 소스를 그대로 복사")

In [ ]:
# Copy Task 학습

# 하이퍼파라미터
vocab_size = 10
d_model = 64
n_heads = 4
d_ff = 128
n_layers = 2
seq_len = 5
batch_size = 64
n_epochs = 100
lr = 0.001

# 모델 생성
model = Transformer(
    src_vocab_size=vocab_size,
    tgt_vocab_size=vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    d_ff=d_ff,
    n_layers=n_layers,
    dropout=0.0  # 작은 task이므로 dropout 없이
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

# 학습 루프
losses = []
model.train()

for epoch in range(n_epochs):
    src, tgt = generate_copy_data(batch_size, seq_len, vocab_size)
    src, tgt = src.to(device), tgt.to(device)
    
    # 입력: tgt[:-1] (SOS + 정답의 앞부분)
    # 정답: tgt[1:]  (정답의 뒷부분, SOS 제외)
    tgt_input = tgt[:, :-1]   # (batch, seq_len)
    tgt_output = tgt[:, 1:]   # (batch, seq_len)
    
    src_mask, tgt_mask = create_masks(src, tgt_input)
    src_mask, tgt_mask = src_mask.to(device), tgt_mask.to(device)
    
    logits = model(src, tgt_input, src_mask, tgt_mask)  # (batch, seq_len, vocab_size)
    
    loss = criterion(logits.reshape(-1, vocab_size), tgt_output.reshape(-1))
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 20 == 0:
        # 정확도 계산
        preds = logits.argmax(dim=-1)
        accuracy = (preds == tgt_output).float().mean()
        print(f"Epoch {epoch+1:>3}/{n_epochs} | Loss: {loss.item():.4f} | Accuracy: {accuracy:.4f}")

In [ ]:
# 학습 곡선 시각화
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Copy Task Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 추론 테스트: greedy decoding

def greedy_decode(model, src, max_len, sos_idx=1):
    """Greedy decoding으로 시퀀스 생성"""
    model.eval()
    with torch.no_grad():
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2).to(device)
        enc_output = model.encode(src, src_mask)
        
        # SOS 토큰으로 시작
        tgt = torch.full((src.size(0), 1), sos_idx, dtype=torch.long).to(device)
        
        for _ in range(max_len):
            tgt_mask = torch.tril(
                torch.ones(tgt.size(1), tgt.size(1), device=device)
            ).bool().unsqueeze(0).unsqueeze(0)
            
            dec_output = model.decode(tgt, enc_output, src_mask, tgt_mask)
            logits = model.output_linear(dec_output[:, -1:])  # 마지막 위치만
            next_token = logits.argmax(dim=-1)  # (batch, 1)
            tgt = torch.cat([tgt, next_token], dim=1)
        
        return tgt[:, 1:]  # SOS 제거


# 테스트
test_src = torch.randint(2, vocab_size, (5, seq_len)).to(device)
predictions = greedy_decode(model, test_src, max_len=seq_len)

print("=== Copy Task 추론 결과 ===")
print(f"{'소스 (입력)':>15} | {'예측 (출력)':>15} | {'일치 여부'}")
print("-" * 55)
correct = 0
for i in range(5):
    src_str = test_src[i].cpu().tolist()
    pred_str = predictions[i].cpu().tolist()
    match = src_str == pred_str
    correct += match
    status = 'O' if match else 'X'
    print(f"{str(src_str):>15} | {str(pred_str):>15} | {status}")

print(f"\n정확도: {correct}/5 ({correct/5*100:.0f}%)")

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: Reverse Task 구현

Copy Task 대신 **Reverse Task**를 구현하세요.
- 입력: `[3, 7, 2, 5]`
- 출력: `[5, 2, 7, 3]` (역순)

위의 학습 코드를 수정해서 reverse task가 잘 학습되는지 확인하세요.

In [ ]:
# TODO: generate_reverse_data 함수 구현
def generate_reverse_data(batch_size, seq_len, vocab_size, pad_idx=0, sos_idx=1):
    """
    Reverse task 데이터 생성
    src: [3, 7, 2, 5]
    tgt: [1, 5, 2, 7, 3]  (SOS + 역순)
    """
    # TODO: 구현하세요
    pass

# TODO: Transformer로 학습하고, 결과를 확인하세요


### 연습 2: 파라미터 수 계산

아래 설정의 Transformer 파라미터 수를 **수동으로** 계산하고, 실제 값과 비교하세요.

```
d_model=512, n_heads=8, d_ff=2048, n_layers=6
src_vocab_size=30000, tgt_vocab_size=30000
```

힌트:
- Embedding: vocab_size * d_model
- Multi-Head Attention: 4 * d_model^2 (W_Q, W_K, W_V, W_O) + biases
- FFN: d_model * d_ff + d_ff + d_ff * d_model + d_model
- LayerNorm: 2 * d_model (gamma, beta)

In [ ]:
# TODO: 수동 계산
d_model = 512
n_heads = 8
d_ff = 2048
n_layers = 6
src_vocab = 30000
tgt_vocab = 30000

# 각 컴포넌트별 파라미터 수를 계산하세요
# src_embedding = ...
# tgt_embedding = ...
# encoder_layer = ...
# decoder_layer = ...
# total = ...

# 실제 모델과 비교
# big_model = Transformer(src_vocab, tgt_vocab, d_model, n_heads, d_ff, n_layers)
# actual = sum(p.numel() for p in big_model.parameters())
# print(f"수동 계산: {total:,}")
# print(f"실제:     {actual:,}")


---
## 핵심 정리

| 컴포넌트 | 역할 | 핵심 수식/포인트 |
|----------|------|------------------|
| Multi-Head Attention | 여러 관점에서 관계 파악 | $d_k = d_{model}/h$, concat 후 $W^O$ |
| FFN | 비선형 변환 추가 | $\text{ReLU}(xW_1+b_1)W_2+b_2$, $d_{ff}=4d_{model}$ |
| LayerNorm | 학습 안정화 | feature 방향 정규화, 배치 무관 |
| Residual Connection | gradient 흐름 보장 | $x + \text{Sublayer}(x)$ |
| Encoder | 입력 표현 생성 | Self-Attn + FFN, N층 |
| Decoder | 출력 생성 | Masked Self-Attn + Cross-Attn + FFN, N층 |
| Causal Mask | 자기회귀 보장 | 하삼각 행렬, 미래 토큰 차단 |

**다음 노트북**: [03-positional-encoding.ipynb](03-positional-encoding.ipynb) - Positional Encoding 심화